In [1]:
import pandas as pd
import polars as pl
import numpy as np
from tqdm.auto import tqdm
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
from pytorch_lightning import LightningModule
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings("ignore")
pd.options.display.max_columns = None
pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(400)
pl.Config.set_fmt_table_cell_list_len(5)

polars.config.Config

In [2]:
class CONFIG:
    seed = 42
    target_col = "responder_6"
    feature_cols = [f"feature_{idx:02d}" for idx in range(79)] + [
        f"responder_{idx}_lag_1" for idx in range(9)
    ]
    model_paths = [
        "./input/js-xs-nn-with-lags",
        "./input/js24-trained-gbdt-model/result.pkl",
    ]

In [3]:
valid = (
    pl.scan_parquet(
        f"./input/js24-preprocessing-create-lags/validation.parquet/",
        n_rows=10000,
    )
    .collect()
    .to_pandas()
)

In [4]:
xgb_model = None
model_path = CONFIG.model_paths[1]
with open(model_path, "rb") as fp:
    result = pickle.load(fp)
    xgb_model = result["model"]
xgb_feature_cols = ["symbol_id", "time_id"] + CONFIG.feature_cols
display(xgb_model)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.01, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=10, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=1000, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [5]:
def r2_val(y_true, y_pred, sample_weight):
    r2 = 1 - np.average((y_pred - y_true) ** 2, weights=sample_weight) / (
        np.average((y_true) ** 2, weights=sample_weight) + 1e-38
    )
    return r2


class NN(LightningModule):
    def __init__(self, input_dim, hidden_dims, dropouts, lr, weight_decay):
        super().__init__()
        self.save_hyperparameters()
        layers = []
        in_dim = input_dim
        for i, hidden_dim in enumerate(hidden_dims):
            layers.append(nn.BatchNorm1d(in_dim))
            if i > 0:
                layers.append(nn.SiLU())
            if i < len(dropouts):
                layers.append(nn.Dropout(dropouts[i]))
            layers.append(nn.Linear(in_dim, hidden_dim))
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, 1))
        layers.append(nn.Tanh())
        self.model = nn.Sequential(*layers)
        self.lr = lr
        self.weight_decay = weight_decay
        self.validation_step_outputs = []

    def forward(self, x):
        return 5 * self.model(x).squeeze(-1)

    def training_step(self, batch):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction="none") * w
        loss = loss.mean()
        self.log(
            "train_loss",
            loss,
            on_step=False,
            on_epoch=True,
            batch_size=x.size(0),
        )
        return loss

    def validation_step(self, batch):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction="none") * w
        loss = loss.mean()
        self.log(
            "val_loss",
            loss,
            on_step=False,
            on_epoch=True,
            batch_size=x.size(0),
        )
        self.validation_step_outputs.append((y_hat, y, w))
        return loss

    def on_validation_epoch_end(self):
        """Calculate validation WRMSE at the end of the epoch."""
        y = (
            torch.cat([x[1] for x in self.validation_step_outputs])
            .cpu()
            .numpy()
        )
        if self.trainer.sanity_checking:
            prob = (
                torch.cat([x[0] for x in self.validation_step_outputs])
                .cpu()
                .numpy()
            )
        else:
            prob = (
                torch.cat([x[0] for x in self.validation_step_outputs])
                .cpu()
                .numpy()
            )
            weights = (
                torch.cat([x[2] for x in self.validation_step_outputs])
                .cpu()
                .numpy()
            )
            # r2_val
            val_r_square = r2_val(y, prob, weights)
            self.log(
                "val_r_square",
                val_r_square,
                prog_bar=True,
                on_step=False,
                on_epoch=True,
            )
        self.validation_step_outputs.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
            self.parameters(), lr=self.lr, weight_decay=self.weight_decay
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=5, verbose=True
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
            },
        }

    def on_train_epoch_end(self):
        if self.trainer.sanity_checking:
            return
        epoch = self.trainer.current_epoch
        metrics = {
            k: v.item() if isinstance(v, torch.Tensor) else v
            for k, v in self.trainer.logged_metrics.items()
        }
        formatted_metrics = {k: f"{v:.5f}" for k, v in metrics.items()}
        print(f"Epoch {epoch}: {formatted_metrics}")

In [6]:
N_folds = 5
models = []
for fold in range(N_folds):
    checkpoint_path = (
        f"{CONFIG.model_paths[0]}/checkpoints/nn_{fold}.model.ckpt"
    )
    models.append(NN.load_from_checkpoint(checkpoint_path))

In [7]:
X_valid = valid[xgb_feature_cols]
y_valid = valid[CONFIG.target_col]
w_valid = valid["weight"]
y_pred_valid_xgb = xgb_model.predict(X_valid)
valid_score = r2_score(y_valid, y_pred_valid_xgb, sample_weight=w_valid)
valid_score

0.8771950006484985

In [8]:
X_valid = valid[CONFIG.feature_cols]
y_valid = valid[CONFIG.target_col]
w_valid = valid["weight"]
X_valid = X_valid.fillna(method="ffill").fillna(0)
X_valid.shape, y_valid.shape, w_valid.shape

((10000, 88), (10000,), (10000,))

In [9]:
y_pred_valid_nn = np.zeros(y_valid.shape)
with torch.no_grad():
    for model in models:
        model.eval()
        y_pred_valid_nn += model(
            torch.FloatTensor(X_valid.values)
        ).cpu().numpy() / len(models)
valid_score = r2_score(y_valid, y_pred_valid_nn, sample_weight=w_valid)
valid_score

0.38665373621545707

In [10]:
y_pred_valid_ensemble = 0.5 * (y_pred_valid_xgb + y_pred_valid_nn)
valid_score = r2_score(y_valid, y_pred_valid_ensemble, sample_weight=w_valid)
valid_score

0.6976754500316179

In [11]:
lags_: pl.DataFrame | None = None

def predict(
    test: pl.DataFrame, lags: pl.DataFrame | None
) -> pl.DataFrame | pd.DataFrame:
    global lags_
    if lags is not None:
        lags_ = lags

    predictions = test.select(
        "row_id",
        pl.lit(0.0).alias("responder_6"),
    )

    if not lags is None:
        lags = lags.group_by(
            ["date_id", "symbol_id"], maintain_order=True
        ).last()
        test = test.join(lags, on=["date_id", "symbol_id"], how="left")
    else:
        test = test.with_columns(
            (pl.lit(0.0).alias(f"responder_{idx}_lag_1") for idx in range(9))
        )

    preds = np.zeros((test.shape[0],))
    preds += xgb_model.predict(test[xgb_feature_cols].to_pandas()) / 2
    test_input = test[CONFIG.feature_cols].to_pandas()
    test_input = test_input.fillna(method="ffill").fillna(0)
    test_input = torch.FloatTensor(test_input.values)
    with torch.no_grad():
        for _, nn_model in enumerate(tqdm(models)):
            nn_model.eval()
            preds += nn_model(test_input).cpu().numpy() / 10

    predictions = test.select("row_id").with_columns(
        pl.Series(
            name="responder_6",
            values=np.clip(preds, a_min=-5, a_max=5),
            dtype=pl.Float64,
        )
    )

    assert isinstance(predictions, pl.DataFrame | pd.DataFrame)
    assert list(predictions.columns) == ["row_id", "responder_6"]
    assert len(predictions) == len(test)
    return predictions

In [12]:
test = pl.read_parquet(
    "./input/jane-street-real-time-market-data-forecasting/test.parquet"
)
lags = pl.read_parquet(
    "./input/jane-street-real-time-market-data-forecasting/lags.parquet"
)
predict(test, lags)

  0%|          | 0/5 [00:00<?, ?it/s]

row_id,responder_6
i64,f64
0,-0.13661
1,-0.328465
2,-0.312823
3,-0.212043
4,-0.048817
5,-0.103214
6,-0.590747
7,-0.067269
8,-0.825635
